In [2]:
"""
==============================================================
SCRIPT 02.7
APLICAR TABLA DE NORMALIZACIÓN

Objetivo
--------
Aplicar automáticamente la tabla de normalización
al Ground Truth bufferizado.

Entradas
---------
GroundTruth_SAGAMI_Buffer5m.gpkg
Tabla_Normalizacion_Clases.csv

Salida
------
GroundTruth_SAGAMI_Buffer5m_Clean.gpkg

==============================================================
"""

from pathlib import Path
import geopandas as gpd
import pandas as pd

# ==========================================================
# CONFIGURACIÓN
# ==========================================================

GT_FILE = Path("/content/GroundTruth_SAGAMI_Buffer5m.gpkg")

TABLE_FILE = Path("/content/Tabla_Normalizacion_Clases.csv")

OUTPUT_FILE = Path("GroundTruth_SAGAMI_Buffer5m_Clean.gpkg")

# ==========================================================
# CARGAR DATOS
# ==========================================================

print("="*70)
print("CARGANDO DATOS")
print("="*70)

gdf = gpd.read_file(GT_FILE)

tabla = pd.read_csv(TABLE_FILE)

print(f"Polígonos : {len(gdf)}")
print(f"Clases en tabla : {len(tabla)}")

# ==========================================================
# VALIDAR TABLA
# ==========================================================

columnas = [
    "clase_original",
    "clase_normalizada"
]

for c in columnas:

    if c not in tabla.columns:
        raise ValueError(f"No existe la columna {c}")

# ==========================================================
# CREAR DICCIONARIO
# ==========================================================

diccionario = dict(
    zip(
        tabla["clase_original"],
        tabla["clase_normalizada"]
    )
)

# ==========================================================
# COPIA DE SEGURIDAD
# ==========================================================

gdf["clase_n3_original"] = gdf["clase_n3"]

# ==========================================================
# APLICAR NORMALIZACIÓN
# ==========================================================

gdf["clase_n3"] = (
    gdf["clase_n3"]
    .astype(str)
    .map(diccionario)
    .fillna(gdf["clase_n3"])
)

# ==========================================================
# REPORTE
# ==========================================================

print("\n")
print("="*70)
print("REPORTE")
print("="*70)

antes = gdf["clase_n3_original"].nunique()
despues = gdf["clase_n3"].nunique()

print(f"Clases antes : {antes}")
print(f"Clases después : {despues}")

print()

print("Cambios realizados")

print("-"*70)

cambios = gdf[
    gdf["clase_n3_original"] != gdf["clase_n3"]
]

if len(cambios)==0:

    print("No hubo cambios.")

else:

    resumen = (
        cambios.groupby(
            [
                "clase_n3_original",
                "clase_n3"
            ]
        )
        .size()
        .reset_index(name="poligonos")
    )

    print(resumen)

# ==========================================================
# CLASES FINALES
# ==========================================================

print("\n")
print("="*70)
print("CLASES FINALES")
print("="*70)

for c in sorted(gdf["clase_n3"].unique()):

    n = (gdf["clase_n3"]==c).sum()

    print(f"{c:50} {n:3}")

# ==========================================================
# GUARDAR
# ==========================================================

gdf.to_file(
    OUTPUT_FILE,
    driver="GPKG"
)

print("\n")
print("="*70)
print("ARCHIVO GENERADO")
print("="*70)

print(OUTPUT_FILE)

print("\nProceso finalizado correctamente.")

CARGANDO DATOS
Polígonos : 67
Clases en tabla : 36


REPORTE
Clases antes : 36
Clases después : 36

Cambios realizados
----------------------------------------------------------------------
No hubo cambios.


CLASES FINALES
0                                                    2
20260105_NVJ_SIMAROUBA2_2025                         1
20260105_NVJ_SIMAROUBA_2025                          1
505730002000000010067000000000                       1
Bosque                                               8
Buffer Bosque                                       12
Buffer laguna                                        1
Cultivo Arroz                                        3
EP_2013                                              1
EP_2017                                              2
EUCALIPTO 2017 - NAVAJAS* CON ANTECEDENTE DE SIEMBRA DE A.M.   1
EUCALIPTO 2017 - NAVAJAS**SIN ANTECEDENTES DE ESTABLECIMIENTO DE PLANTACION   1
EU_2017_20 has                                       1
EU_2017_7.1 has           